# MLflow Training Monitoring

## Objectifs d'apprentissage
- Travailler avec un dataset réel de Kaggle (House Prices - Advanced Regression Techniques)
- Entraîner un modèle de régression
- Monitorer les expériences avec MLflow
- Utiliser l'interface MLflow Tracking UI pour comparer les exécutions, métriques et artefacts

## Instructions
Suivez les 7 étapes pour compléter ce défi

In [ ]:
# Install required libraries
!pip install mlflow scikit-learn pandas matplotlib seaborn -q

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request

# ETAPE 1: Setup MLflow
print("=== ETAPE 1: Configuration MLflow ===")
mlflow.set_tracking_uri("file:./mlruns")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
experiment_name = "MLflow_House_Prices_Experiment"
mlflow.set_experiment(experiment_name)
print(f"Experiment: {experiment_name}\n")

# ETAPE 2: Load and explore dataset
print("=== ETAPE 2: Chargement du dataset ===")
url = "https://github.com/ageron/handson-ml2/raw/master/datasets/housing/housing.csv"
housing_data = pd.read_csv(url)
print(f"Dataset shape: {housing_data.shape}")
print(f"Columns: {housing_data.columns.tolist()}")
print(f"Missing values:\n{housing_data.isnull().sum()}")
print(f"\nDataset preview:\n{housing_data.head()}\n")

# ETAPE 3: Data preprocessing
print("=== ETAPE 3: Pretraitement des donnees ===")
data_clean = housing_data.dropna()
print(f"Clean dataset shape: {data_clean.shape}")

feature_columns = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
X = data_clean[feature_columns]
y = data_clean['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}\n")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features scaled using StandardScaler\n")

# ETAPE 4: MLflow Tracking Basics
print("=== ETAPE 4: MLflow Tracking Basics ===")
print("MLflow tracking URI configured")
print("Ready to log experiments and metrics\n")

# ETAPE 5 & 6: Train multiple models and log experiments
print("=== ETAPE 5 & 6: Entrainement et logging des experiments ===")

# Run 1: Linear Regression
with mlflow.start_run(run_name="LinearRegression_Run1"):
    print("\nRun 1: Linear Regression - Baseline")
    lr_model = LinearRegression()
    lr_model.fit(X_train_scaled, y_train)

    y_pred_train = lr_model.predict(X_train_scaled)
    y_pred_test = lr_model.predict(X_test_scaled)

    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    test_mae = mean_absolute_error(y_test, y_pred_test)

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("features_count", len(feature_columns))
    mlflow.log_param("test_size", 0.2)

    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("test_mae", test_mae)

    mlflow.sklearn.log_model(lr_model, "model")

    print(f"  Train RMSE: {train_rmse:.2f}")
    print(f"  Test RMSE: {test_rmse:.2f}")
    print(f"  Test R2: {test_r2:.4f}")

# Run 2: Random Forest - 50 estimators
with mlflow.start_run(run_name="RandomForest_Run1"):
    print("\nRun 2: Random Forest - 50 estimators")
    rf_model = RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_scaled, y_train)

    y_pred_train = rf_model.predict(X_train_scaled)
    y_pred_test = rf_model.predict(X_test_scaled)

    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    test_mae = mean_absolute_error(y_test, y_pred_test)

    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 50)
    mlflow.log_param("max_depth", 15)
    mlflow.log_param("features_count", len(feature_columns))

    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("test_mae", test_mae)

    mlflow.sklearn.log_model(rf_model, "model")

    print(f"  Train RMSE: {train_rmse:.2f}")
    print(f"  Test RMSE: {test_rmse:.2f}")
    print(f"  Test R2: {test_r2:.4f}")

# Run 3: Random Forest - 100 estimators
with mlflow.start_run(run_name="RandomForest_Run2"):
    print("\nRun 3: Random Forest - 100 estimators")
    rf_model_2 = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
    rf_model_2.fit(X_train_scaled, y_train)

    y_pred_train = rf_model_2.predict(X_train_scaled)
    y_pred_test = rf_model_2.predict(X_test_scaled)

    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    test_mae = mean_absolute_error(y_test, y_pred_test)

    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 20)
    mlflow.log_param("features_count", len(feature_columns))

    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)
    mlflow.log_metric("test_mae", test_mae)

    mlflow.sklearn.log_model(rf_model_2, "model")

    print(f"  Train RMSE: {train_rmse:.2f}")
    print(f"  Test RMSE: {test_rmse:.2f}")
    print(f"  Test R2: {test_r2:.4f}")

print("\n=== ETAPE 7: Comparaison des experiences ===")
print("All experiments logged successfully!")
print("Access MLflow UI with: mlflow ui")